# 05 — Clasificación, árboles y ensambles

## Motivación

Hasta ahora el objetivo $y$ era un número continuo: un precio, un valor de $\sin(x)$.
Muchos problemas tienen un objetivo categórico: ¿este tumor es maligno o benigno?,
¿esta conexión de red es un ataque?, ¿este correo es spam? Ese es un problema de
**clasificación**.

Esta sesión cubre dos familias de modelos de clasificación (vecinos más cercanos y
modelos lineales; luego, árboles y ensambles de árboles) y las métricas para
evaluarlos correctamente, tema pendiente de la sesión 04. Un solo dataset, **Breast
Cancer Wisconsin** (diagnóstico de tumores a partir de mediciones de una biopsia),
sirve de hilo conductor para no perder tiempo de clase en contexto nuevo cada vez que
cambia el modelo.

In [ ]:
import matplotlib.pyplot as plt  # excepción documentada: plot_tree y ConfusionMatrixDisplay
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(as_frame=True)
X, y = cancer.data, cancer.target

print(f"shape: {X.shape}")
print(f"clases: {dict(zip(cancer.target_names, np.bincount(y)))}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

`stratify=y` mantiene la misma proporción de clases en train y test que en el dataset
completo. Sin esto, una división al azar podría dejar, por accidente, más tumores
malignos en un conjunto que en el otro.

## 1. K vecinos más cercanos (KNN)

El modelo de clasificación más directo: para predecir la clase de un punto nuevo, se
buscan sus $k$ vecinos más cercanos en el conjunto de entrenamiento (por distancia
euclidiana) y se asigna la clase mayoritaria entre ellos.

$$d(\mathbf{x}_i, \mathbf{x}_j) = \sqrt{\sum_{l=1}^{d} (x_{il} - x_{jl})^2}$$

KNN memoriza el conjunto de entrenamiento y calcula distancias en el momento de
predecir, sin ajustar parámetros. La distancia depende directamente de la escala de
cada característica: una con rango numérico grande (`mean area`, en cientos) domina
la suma sobre una con rango pequeño (`mean smoothness`, entre 0 y 1), sin importar
cuál sea más informativa. KNN **requiere** escalar las características a un rango
común.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

knn_sin_escalar = KNeighborsClassifier(n_neighbors=5)
knn_sin_escalar.fit(X_train, y_train)

knn_escalado = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn_escalado.fit(X_train, y_train)

print(f"KNN sin escalar:  accuracy = {knn_sin_escalar.score(X_test, y_test):.4f}")
print(f"KNN escalado:     accuracy = {knn_escalado.score(X_test, y_test):.4f}")

## 2. Regresión logística

La regresión lineal predice el valor $z = \mathbf{w}^\top\mathbf{x} + b$ directamente,
sin restricción de rango: $z \in \mathbb{R}$. Clasificación binaria requiere una
probabilidad, acotada en $(0, 1)$. La regresión logística evalúa la función
**sigmoide** en ese predictor lineal:

$$P(y=1 \mid \mathbf{x}) = \sigma(z) = \sigma(\mathbf{w}^\top\mathbf{x} + b)
= \frac{1}{1 + e^{-(\mathbf{w}^\top\mathbf{x} + b)}}$$

$\sigma$ es monótona creciente, con $\sigma(0) = 0.5$ y asíntotas en $0$ y en $1$
conforme $z \to \mp\infty$: valores grandes y positivos de $z$ dan probabilidades
cercanas a $1$; valores grandes y negativos, cercanas a $0$.

In [ ]:
z = np.linspace(-10, 10, 400)
sigma = 1 / (1 + np.exp(-z))

fig = go.Figure()
fig.add_trace(go.Scatter(x=z, y=sigma, mode="lines", name="σ(z)", line=dict(width=2.5)))
fig.add_trace(go.Scatter(
    x=[z.min(), z.max()], y=[0.5, 0.5], mode="lines", name="umbral σ = 0.5",
    line=dict(width=1.5, dash="dash", color="gray"),
))
fig.add_trace(go.Scatter(
    x=[0, 0], y=[0, 1], mode="lines", showlegend=False,
    line=dict(width=1.5, dash="dash", color="gray"),
))
fig.update_layout(
    title="Función sigmoide",
    xaxis_title="z",
    yaxis_title="σ(z)",
    template="plotly_white",
    width=550, height=400,
)
fig.show()

La regla de decisión usual: predecir clase 1 si $P(y=1\mid\mathbf{x}) > 0.5$,
equivalente a $z > 0$. Los coeficientes siguen siendo interpretables, ahora en la
escala de **log-odds**:

$$\log\frac{P(y=1\mid\mathbf{x})}{1 - P(y=1\mid\mathbf{x})} = \mathbf{w}^\top\mathbf{x} + b$$

un aumento de una unidad en $x_j$ (con las demás fijas) suma $w_j$ al log-odds de la
clase 1.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1]

print(f"regresión logística:  accuracy = {logreg.score(X_test, y_test):.4f}")

## 3. Métricas de clasificación

La sesión 04 cerró con un caso donde la accuracy no distingue un modelo útil de uno
que ignora la clase minoritaria. La **matriz de confusión** desglosa los cuatro casos
posibles frente a una clase de referencia ("positiva"):

| | predicho negativo | predicho positivo |
|---|---|---|
| **real negativo** | verdadero negativo (TN) | falso positivo (FP) |
| **real positivo** | falso negativo (FN) | verdadero positivo (TP) |

De ahí se calculan métricas específicas:

$$\text{precisión} = \frac{TP}{TP + FP}, \qquad
\text{exhaustividad (recall)} = \frac{TP}{TP + FN}, \qquad
F_1 = \frac{2 \cdot \text{precisión} \cdot \text{recall}}{\text{precisión} + \text{recall}}$$

**Precisión** responde "de lo que el modelo marcó como positivo, cuánto era correcto";
**recall**, "de lo que era positivo en realidad, cuánto encontró el modelo". En el
caso de KDD Cup 99 de la sesión 04, un modelo que nunca predice "ataque" tiene recall
cero en la clase ataque: invisible para la accuracy, evidente en el recall.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

print(classification_report(y_test, y_pred, target_names=cancer.target_names))

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=cancer.target_names, ax=ax, colorbar=False
)
ax.set_title("Matriz de confusión — regresión logística")
plt.show()

El umbral de decisión (0.5 por defecto) es ajustable: bajarlo aumenta el recall a
costa de la precisión, y viceversa. La curva **ROC** grafica esa disyuntiva completa:
tasa de verdaderos positivos contra tasa de falsos positivos, para todos los umbrales
posibles. El área bajo la curva (**AUC**) resume la curva en un número: 1.0 es un
clasificador perfecto, 0.5 equivale a decidir al azar.

In [ ]:
from sklearn.metrics import auc, roc_curve

fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_score = auc(fpr, tpr)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr, mode="lines", name=f"regresión logística (AUC = {auc_score:.3f})",
    line=dict(width=2.5),
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="clasificador al azar",
    line=dict(width=1.5, dash="dash", color="gray"),
))
fig.update_layout(
    title="Curva ROC",
    xaxis_title="tasa de falsos positivos",
    yaxis_title="tasa de verdaderos positivos",
    template="plotly_white",
    width=550, height=500,
)
fig.show()

## 4. Árboles de decisión

KNN y la regresión logística comparten un requisito: características en una escala
común. Un **árbol de decisión** no comparte ese requisito: cada partición compara una
sola característica contra un umbral (`mean radius <= 14.3`), sin combinar escalas
distintas en una misma operación.

El árbol se construye de forma voraz: en cada nodo, se selecciona la característica y
el umbral que producen la partición más "pura" posible. La impureza de Gini de un nodo
con $K$ clases y proporciones $p_1, \ldots, p_K$ es

$$G = 1 - \sum_{k=1}^{K} p_k^2$$

$G = 0$ cuando el nodo es puro (una sola clase); $G$ es máxima cuando las clases están
repartidas en partes iguales. Cada partición se elige para minimizar la impureza
promedio (ponderada por tamaño) de los dos nodos resultantes.

Para ver un árbol legible se usa **Wine**: 3 clases, 13 características con nombre,
un tamaño que cabe en un diagrama.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.tree import DecisionTreeClassifier, plot_tree

wine = load_wine(as_frame=True)
Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42, stratify=wine.target
)

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(Xw_train, yw_train)
print(f"árbol (max_depth=3):  accuracy = {tree.score(Xw_test, yw_test):.4f}")

fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    tree, feature_names=wine.feature_names, class_names=wine.target_names,
    filled=True, fontsize=9, ax=ax,
)
plt.show()

Cada nodo muestra la condición de partición, el valor de `gini`, cuántas muestras de
train llegaron a ese nodo y cómo se reparten entre clases. Un árbol sin límite de
profundidad puede crecer hasta tener un nodo puro por cada punto de entrenamiento: es,
en árboles, el equivalente al polinomio de grado 15 de la sesión 03, sobreajuste por
exceso de flexibilidad. `max_depth` es la forma más directa de controlarlo.

## 5. Ensambles: combinar muchos árboles

Un árbol individual, sin restricción de profundidad, tiene varianza alta: pequeños
cambios en los datos de entrenamiento cambian mucho su estructura. Los ensambles
combinan muchos árboles para reducir ese error, por dos caminos distintos:

- **Bagging** (*bootstrap aggregating*), implementado en scikit-learn como **Random
  Forest**: entrena muchos árboles independientes, cada uno sobre una muestra
  bootstrap de los datos (con reemplazo) y un subconjunto aleatorio de características
  en cada partición, y promedia sus predicciones. Promediar modelos de varianza alta
  pero sesgo bajo reduce la varianza del conjunto, siguiendo la misma descomposición
  sesgo-varianza de la sesión 03.
- **Boosting**, implementado como **Gradient Boosting**: entrena árboles **en
  secuencia**, cada uno ajustando los errores del anterior. Reduce el sesgo del
  conjunto, a costa de mayor riesgo de sobreajuste si se entrenan demasiados árboles.

Ambos heredan de los árboles la independencia de escala: no necesitan
`StandardScaler`.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)

print(f"regresión logística:  accuracy = {logreg.score(X_test, y_test):.4f}")
print(f"random forest:         accuracy = {rf.score(X_test, y_test):.4f}")
print(f"gradient boosting:     accuracy = {gb.score(X_test, y_test):.4f}")

En este dataset los tres modelos rinden de forma similar: Breast Cancer Wisconsin es
casi linealmente separable, así que la regresión logística ya captura la mayor parte
de la estructura. La ventaja de los ensambles se nota más en datasets con relaciones
no lineales entre características, como el ejercicio 3 de esta sesión.

### Importancia de características

Un árbol reporta cuánto contribuyó cada característica a reducir la impureza,
acumulado sobre todas las particiones del bosque: la **importancia de
características**, distinta de la dirección de efecto que da un coeficiente lineal.

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=cancer.feature_names).sort_values().tail(10)

fig = go.Figure()
fig.add_trace(go.Bar(x=importancias.values, y=importancias.index, orientation="h"))
fig.update_layout(
    title="Importancia de características — Random Forest (top 10)",
    xaxis_title="reducción de impureza acumulada",
    template="plotly_white",
)
fig.show()

## Ejercicio

Trabaja en una copia de este notebook dentro de `mi-trabajo/`.

1. **El valor de $k$ en KNN.** Sobre Breast Cancer Wisconsin (con escalado), calcula la
   accuracy en test para $k \in \{1, 3, 5, 7, 9, 15, 25\}$ y grafica el resultado con
   Plotly. ¿Qué le pasa a $k=1$? Conecta tu respuesta con sesgo y varianza.

2. **Profundidad del árbol y sobreajuste.** Entrena árboles de decisión con
   `max_depth` de 1 a 15 sobre Breast Cancer Wisconsin y grafica accuracy de train y
   de test contra la profundidad. Es la misma curva de validación de la sesión 03,
   con un hiperparámetro distinto.

3. **Reto — ensambles a escala real.** Carga `sklearn.datasets.fetch_covtype` (más de
   500,000 filas, 7 clases de cobertura forestal). Toma una muestra de 20,000 filas
   (`df.sample(20_000, random_state=42)`) para que el entrenamiento sea razonable en
   clase, entrena un `RandomForestClassifier` y reporta accuracy y la matriz de
   confusión de 7×7. ¿Qué clases confunde más el modelo entre sí?